[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ha466/kokoro_nano/blob/main/colab_training.ipynb)

# Kokoro-7M 2-Voice (Female + Male) Distillation on Google Colab (T4 GPU)

This notebook trains a compact **7.48M parameter Kokoro student model** conditioned on two voices:
- **Female Voice**: `af_bella`
- **Male Voice**: `am_adam`

Estimated training time on a free Colab **T4 GPU**: ~45–60 minutes for 20,000 steps.

### Step 0: Clone Repository & Setup Workspace
If you opened this notebook directly in Google Colab, this cell automatically clones the repository and enters the workspace directory.


In [ ]:
# Clone repository if running in a fresh Colab instance
import os
if not os.path.exists('training'):
    !git clone https://github.com/ha466/kokoro_nano.git
    %cd kokoro_nano
!pwd


### Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies

In [ ]:
%pip install -r requirements.txt
!python -m spacy download en_core_web_sm

### Step 3: Choose Your Training Mode & Generate Dataset

> **Option A: Quick Test (20 sentences)** — Generates in 30 seconds to test that everything runs.
>
> **Option B: Full Training (3,000 to 10,000 sentences)** — Recommended for a production model that generalizes to any new sentence without robotic artifacts.

#### Option A: Quick Pipeline Test (20 Sample Sentences)

In [ ]:
# Quick sanity check (20 sentences, ~30s on T4)
!python training/generate_teacher_dataset.py \
    --female-voice af_bella \
    --male-voice am_adam \
    --out-dir dist_twovoice

#### Option B: Full Training (Recommended: 3,000+ Sentences)
Downloads clean open-domain sentences automatically and synthesizes speech with 50% female and 50% male voices.

In [ ]:
# 1. Fetch 3,000 clean sentences (or upload your own .txt file)
!python training/fetch_sentences.py --count 3000 --out sentences.txt

# 2. Generate teacher paired audio & durations
!python training/generate_teacher_dataset.py \
    --texts sentences.txt \
    --female-voice af_bella \
    --male-voice am_adam \
    --out-dir dist_twovoice

### Step 4: Train the 7.48M Student Model
Trains with:
- Multi-resolution STFT loss
- Duration L1 loss matching teacher alignment
- Log-mel L1 loss (`--mel-weight 5.0`)
- Silence loss (`--sil-weight 100.0`)
- Multi-Period & Multi-Resolution Spectrogram Discriminators (MPD + MSD)
- Cosine LR decay

In [ ]:
!python training/train_student.py \
    --index dist_twovoice/index.jsonl \
    --bin dist_twovoice/audio.i16.bin \
    --config config.json \
    --two-voices \
    --batch 16 \
    --steps 20000 \
    --mel-weight 5.0 \
    --sil-weight 100.0 \
    --lr 2e-4 \
    --lr-decay \
    --out runs/twovoice_student

### Step 5: Test Inference & Play Audio
Listen to your trained 2-voice model directly in Colab!

In [ ]:
import torch
from IPython.display import Audio, display
from load_model import load

# 1. Load trained student checkpoint
weights_path = "runs/twovoice_student/student.pth"
model, pipeline, _ = load(weights=weights_path, config="config.json")

# 2. Load the two voice packs trained with the model
female_voice = torch.load("runs/twovoice_student/voice_female.pt", weights_only=True)
male_voice   = torch.load("runs/twovoice_student/voice_male.pt", weights_only=True)

# 3. Synthesize Female Voice
text_female = "Hello! This is the newly trained female voice running on our compact model."
audio_f = next(pipeline(text_female, voice=female_voice))[2]
print("[+] Female Voice:")
display(Audio(audio_f, rate=24000))

# 4. Synthesize Male Voice
text_male = "And this is the male voice, generated by the exact same lightweight model."
audio_m = next(pipeline(text_male, voice=male_voice))[2]
print("[+] Male Voice:")
display(Audio(audio_m, rate=24000))

### Step 6: Download Complete Model Bundle
Packages `student.pth` (weights) + `voice_female.pt` + `voice_male.pt` (voice packs) + `config.json` into a single `twovoice_model.zip` file.

In [ ]:
!zip -j twovoice_model.zip \
    runs/twovoice_student/student.pth \
    runs/twovoice_student/voice_female.pt \
    runs/twovoice_student/voice_male.pt \
    runs/twovoice_student/config.json

from google.colab import files
files.download("twovoice_model.zip")